# GPAT

Gridded Plume Analysis Tool (GPAT) modelling framework. This simulates flight trajectories, estimates fuel burn and emissions, models dispersion effects, and aggregates plume data to a common Eulerian grid for further photochemical and microphysical processing.

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from dataclasses import asdict
from pycontrails.models.gpat.gpat import GPAT, SimParams, FlParams, PlParams, MetParams, ChemParams, dict_to_dataclass
import os
import holoviews as hv
import hvplot.pandas
import hvplot.xarray

In [2]:
# global simulation parameters
sim_params = {
    "t_fl": (pd.to_datetime("2022-01-20 13:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=1)),# (start time, time step, run time)
    "t_pl": (pd.to_datetime("2022-01-20 13:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=2)),# (start time, time step, max age)
    "t_sim": (pd.to_datetime("2022-01-20 12:00:00"), pd.Timedelta(seconds=20), pd.Timedelta(hours=4)),# (start time, time step, run time)
    "t_out": (pd.to_datetime("2022-01-20 12:00:00"), pd.Timedelta(minutes=5), pd.Timedelta(hours=4)),# (start time, time step, run time)
    "lat_bounds": (0.0, 1.0),  # lat bounds [deg]
    "lon_bounds": (0.0, 1.0),  # lon bounds [deg]
    "alt_bounds": (10000, 11000),  # alt bounds [m]
    "hres_sim_c": 0.05,  # coarse horizontal resolution [deg]
    "vres_sim_c": 500,  # coarse vertical resolution [m]
    "hres_sim_f": 0.001,  # fine horizontal resolution [deg]
    "vres_sim_f": 100,  # fine vertical resolution [m]

    "run_path": "/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/",
    "data_path": "/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/", # "/projects/Impact_of_aviation_on_climate
    "job_id": "GPAT_Jan_2026_test_2_ac",
}

In [3]:
#flight trajectory parameters
fl_params = {
    "mode": "synthetic",
    "file": None,  # flight trajectory file

    "ac_type": "A320",  # aircraft type
    "fl0_speed": 100.0,  # m/s
    "fl0_heading": 45.0,  # deg
    "fl0_coords0": (0.1, 0.1, 10500),  # lat, lon, alt [deg, deg, m]
    "sep_dist": (10000, 5000, 0),  # dx, dy, dz [m]
    "n_ac": 2,  # number of aircraft
}

In [4]:
# plume dispersion parameters
pl_params = {
    "depth": 50.0,  # initial plume depth, [m]
    "width": 50.0,  # initial plume width, [m]
    "verbose_outputs": False,  # print verbose outputs
    "n_slices": 3,  # number of slices in the plume
    "f_max": 0.99,  # maximum fraction of total emissions in any slice
    "shear": 0.01,  # shear [m/s]
    }

In [5]:
# meteorology parameters
met_params = {
    "eastward_wind": 5.0,  # m/s
    "northward_wind": 3.0,  # m/s
    "lagrangian_tendency_of_air_pressure": 0.0,  # m/s
}

In [6]:
# chemistry parameters
chem_params = {
    "run_chem": True,
    "species_emi": ("NO",),
    "species_pl": ("NO", "NO2", "O3", "NO3", "N2O5",
                      "HNO3", "HONO", "HO2NO2","PAN", 
                      "CH3O2NO2","H2O2", "CH3OOH",
                      "CO", "CH4", "HCHO", "SO2", "SA"),
    "species_out": ("O3", "NO2", "NO", "NO3", "N2O5", 
                    "HNO3", "HONO", "HO2", "OH", "H2O2",
                    "CO", "CH4", "CH3O2","HO2NO2", "PAN", "SO2" )
}

In [7]:
sim_params = SimParams(**sim_params)
fl_params = FlParams(**fl_params)
pl_params = PlParams(**pl_params)
met_params = MetParams(**met_params)
chem_params = ChemParams(**chem_params)

gpat = GPAT(sim_params, fl_params, pl_params, met_params, chem_params)

In [8]:
gpat.preprocess_gpat()

/home/ktait98/miniconda3/envs/contrails/lib/python3.12/site-packages/xarray/core/duck_array_ops.py:234: UserWarning: no explicit representation of timezones available for np.datetime64
  return data.astype(dtype, **kwargs)


flight 0 done
flight 1 done


/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:618: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  fl[i].dataframe[column] = fl[i].dataframe[column].fillna(method="ffill")
/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:681: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  fl[i][column] = fl[i][column].fillna(method="ffill")
/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:681: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  fl[i][column] = fl[i][column].fillna(method="ffill")
/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:872: RuntimeWarning: invalid value encountered in cast
  age_seconds = np.where(np.isnat(age_values), 0, (age_values / np

Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/inputs/GPAT_Jan_2026_test_2_ac/boxm_ds.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/inputs/GPAT_Jan_2026_test_2_ac/fl_ds.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/inputs/GPAT_Jan_2026_test_2_ac/pl_ds.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/outputs/GPAT_Jan_2026_test_2_ac/boxm_out.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/outputs/GPAT_Jan_2026_test_2_ac/patch_table.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/outputs/GPAT_Jan_2026_test_2_ac/pl_out.nc


In [9]:
gpat.eval()

 FL_DS SUMMARY:
   NSEG =           45
   IS_OPEN =  T
   NCID =        65536
 PL_DS SUMMARY:
   NSEG =           44
   NSEMI =            1
   NTPL =          134
   IS_OPEN =  T
   NCID =       131072
 BOXM_DS SUMMARY:
   NCELL =          800
   NSBOXM =          219
   NTBOXM =          721
   IS_OPEN =  T
   NCID =       196608


STOP PATCH_TABLE_WRITE: PATCH_STATE%Y_DEL_F NOT ALLOCATED


In [10]:
fl_ds = xr.open_dataset(f"{gpat.inputs_job}/fl_ds.nc")
fl_ds

<xarray.Dataset> Size: 10kB
Dimensions:               (seg_id: 45)
Coordinates:
    flight_id             (seg_id) int64 360B ...
    waypoint              (seg_id) int64 360B ...
  * seg_id                (seg_id) int64 360B 0 1 2 3 4 5 ... 39 40 41 42 43 44
Data variables: (12/16)
    longitude             (seg_id) float64 360B ...
    latitude              (seg_id) float64 360B ...
    level                 (seg_id) float64 360B ...
    altitude              (seg_id) float64 360B ...
    time                  (seg_id) <U20 4kB ...
    true_airspeed         (seg_id) float64 360B ...
    ...                    ...
    thrust                (seg_id) float64 360B ...
    rocd                  (seg_id) float64 360B ...
    fuel_flow_per_engine  (seg_id) float64 360B ...
    thrust_setting        (seg_id) float64 360B ...
    time_rel_s            (seg_id) int64 360B ...
    time_idx              (seg_id) int64 360B ...
Attributes:
    description:  Flight trajectory and emissions data for BOXM

In [11]:
pl_ds = xr.open_dataset(f"{gpat.inputs_job}/pl_ds.nc")
pl_ds


<xarray.Dataset> Size: 2MB
Dimensions:          (seg_id: 44, time: 134, species_emi: 1, ht: 2)
Coordinates:
  * seg_id           (seg_id) int64 352B 0 1 2 3 4 5 6 ... 37 38 39 40 41 42 43
  * time             (time) <U20 11kB '2022-01-20T13:00:00Z' ... '2022-01-20T...
  * species_emi      (species_emi) <U2 8B 'NO'
    flight_id        (seg_id) int64 352B ...
    waypoint         (seg_id) int64 352B ...
    species_emi_num  (species_emi) int64 8B ...
    time_rel_s       (time) int64 1kB ...
    time_idx         (time) int64 1kB ...
  * ht               (ht) <U4 32B 'tail' 'head'
Data variables: (12/13)
    age              (seg_id, time) <U21 495kB ...
    longitude        (seg_id, ht, time) float64 94kB ...
    latitude         (seg_id, ht, time) float64 94kB ...
    level            (seg_id, ht, time) float64 94kB ...
    width            (seg_id, ht, time) float64 94kB ...
    depth            (seg_id, ht, time) float64 94kB ...
    ...               ...
    sigma_yy         (seg_id, ht, time) float64 94kB ...
    sigma_yz         (seg_id, ht, time) float64 94kB ...
    sigma_zz         (seg_id, ht, time) float64 94kB ...
    altitude         (seg_id, ht, time) float64 94kB ...
    emi_pl_mass      (seg_id, species_emi) float64 352B ...
    age_s            (seg_id, time) int64 47kB ...
Attributes:
    nseg:             43
    ts_fl:            60.0
    ts_pl:            60.0
    ts_sim:           20.0
    ts_out:           300.0
    species_emi:      NO
    species_pl:       ['NO', 'NO2', 'O3', 'NO3', 'N2O5', 'HNO3', 'HONO', 'HO2...
    species_emi_num:  8
    species_pl_num:   [  8   4   6   5   7  14  13  15 198 217  12 144  11  2...
    n_slices:         3
    f_max:            0.99
    description:      Emission species mass in plume segments

In [12]:
boxm_ds = xr.open_dataset(f"{gpat.inputs_job}/boxm_ds.nc")

boxm_ds

<xarray.Dataset> Size: 29MB
Dimensions:           (time: 721, cell: 800, species_boxm: 219)
Coordinates:
  * time              (time) <U20 58kB '2022-01-20T12:00:00Z' ... '2022-01-20...
    air_pressure      (cell) float64 6kB ...
    altitude_c        (cell) float64 6kB ...
  * species_boxm      (species_boxm) <U10 9kB 'O1D' 'O' 'OH' ... 'EMPOA' 'P2007'
    time_rel_s        (time) int64 6kB ...
    time_idx          (time) int64 6kB ...
    species_boxm_num  (species_boxm) int64 2kB ...
    level_c           (cell) float64 6kB ...
    longitude_c       (cell) float64 6kB ...
    latitude_c        (cell) float64 6kB ...
Dimensions without coordinates: cell
Data variables:
    air_temperature   (cell, time) float64 5MB ...
    H2O               (cell, time) float64 5MB ...
    M                 (cell, time) float64 5MB ...
    O2                (cell, time) float64 5MB ...
    N2                (cell, time) float64 5MB ...
    sza               (cell, time) float64 5MB ...
    Y_bg_c            (cell, species_boxm) float64 1MB ...
Attributes: (12/14)
    ts_fl:          60.0
    ts_pl:          60.0
    ts_sim:         20.0
    ts_out:         300.0
    hres_sim_c:     0.05
    vres_sim_c:     500
    ...             ...
    photol_params:  57
    photol_coeffs:  96
    therm_coeffs:   512
    flux_species:   130
    description:    BOXM coarse-grid meteorology and background chemistry fields
    note:           Emissions and plume segments handled separately via PL_DS...

In [13]:
#from IPython.display import clear_output

pl_out = xr.open_dataset(f"{gpat.outputs_job}/pl_out.nc")
pl_out


<xarray.Dataset> Size: 282kB
Dimensions:          (seg_id: 44, time: 49, species_out: 16)
Coordinates:
  * seg_id           (seg_id) int64 352B 0 1 2 3 4 5 6 ... 37 38 39 40 41 42 43
    flight_id        (seg_id) int64 352B ...
    waypoint         (seg_id) int64 352B ...
  * time             (time) <U20 4kB '2022-01-20T12:00:00Z' ... '2022-01-20T1...
  * species_out      (species_out) <U6 384B 'O3' 'NO2' 'NO' ... 'PAN' 'SO2'
    species_out_num  (species_out) int64 128B ...
    time_rel_s       (time) int64 392B ...
    time_idx         (time) int64 392B ...
Data variables:
    pl_mass          (seg_id, species_out, time) float64 276kB ...
Attributes:
    description:  Plume segment output for BOXM

In [18]:
from IPython.display import clear_output
clear_output(wait=True)

pl_out.pl_mass.sel(seg_id=0).values
#pl_out.pl_mass.where(pl_out["time_idx"] == 1).sel(seg_id=0, species_out='NO').values

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 

In [15]:
from IPython.display import clear_output
clear_output(wait=True)

boxm_out = xr.open_dataset(f"{gpat.outputs_job}/boxm_out.nc")

boxm_out["time_idx"].values

array([  1,  16,  31,  46,  61,  76,  91, 106, 121, 136, 151, 166, 181,
       196, 211, 226, 241, 256, 271, 286, 301, 316, 331, 346, 361, 376,
       391, 406, 421, 436, 451, 466, 481, 496, 511, 526, 541, 556, 571,
       586, 601, 616, 631, 646, 661, 676, 691, 706, 721])

In [16]:
gpat.patch_table

<xarray.Dataset> Size: 512B
Dimensions:          (row: 0, species_out: 16)
Coordinates:
  * row              (row) int64 0B 
    patch_id         (row) int64 0B 
  * species_out      (species_out) <U6 384B 'O3' 'NO2' 'NO' ... 'PAN' 'SO2'
    species_out_num  (species_out) int64 128B 6 4 8 5 7 14 ... 21 22 15 198 16
    time             (row) object 0B 
    time_rel_s       (row) int64 0B 
    time_idx         (row) int64 0B 
    latitude_f       (row) float64 0B 
    longitude_f      (row) float64 0B 
    altitude_f       (row) float64 0B 
    level_f          (row) float64 0B 
Data variables:
    Y_del_f          (row, species_out) float64 0B dask.array<chunksize=(0, 16), meta=np.ndarray>
Attributes:
    description:  Fine grid plume patch output for BOXM

In [17]:
gpat.pl_out["time"]

<xarray.DataArray 'time' (time: 49)> Size: 392B
array(['2022-01-20T12:00:00Z', '2022-01-20T12:05:00Z', '2022-01-20T12:10:00Z',
       '2022-01-20T12:15:00Z', '2022-01-20T12:20:00Z', '2022-01-20T12:25:00Z',
       '2022-01-20T12:30:00Z', '2022-01-20T12:35:00Z', '2022-01-20T12:40:00Z',
       '2022-01-20T12:45:00Z', '2022-01-20T12:50:00Z', '2022-01-20T12:55:00Z',
       '2022-01-20T13:00:00Z', '2022-01-20T13:05:00Z', '2022-01-20T13:10:00Z',
       '2022-01-20T13:15:00Z', '2022-01-20T13:20:00Z', '2022-01-20T13:25:00Z',
       '2022-01-20T13:30:00Z', '2022-01-20T13:35:00Z', '2022-01-20T13:40:00Z',
       '2022-01-20T13:45:00Z', '2022-01-20T13:50:00Z', '2022-01-20T13:55:00Z',
       '2022-01-20T14:00:00Z', '2022-01-20T14:05:00Z', '2022-01-20T14:10:00Z',
       '2022-01-20T14:15:00Z', '2022-01-20T14:20:00Z', '2022-01-20T14:25:00Z',
       '2022-01-20T14:30:00Z', '2022-01-20T14:35:00Z', '2022-01-20T14:40:00Z',
       '2022-01-20T14:45:00Z', '2022-01-20T14:50:00Z', '2022-01-20T14:55:00Z',
       '2022-01-20T15:00:00Z', '2022-01-20T15:05:00Z', '2022-01-20T15:10:00Z',
       '2022-01-20T15:15:00Z', '2022-01-20T15:20:00Z', '2022-01-20T15:25:00Z',
       '2022-01-20T15:30:00Z', '2022-01-20T15:35:00Z', '2022-01-20T15:40:00Z',
       '2022-01-20T15:45:00Z', '2022-01-20T15:50:00Z', '2022-01-20T15:55:00Z',
       '2022-01-20T16:00:00Z'], dtype=object)
Coordinates:
  * time        (time) object 392B '2022-01-20T12:00:00Z' ... '2022-01-20T16:...
    time_rel_s  (time) int64 392B 0 300 600 900 1200 ... 13500 13800 14100 14400
    time_idx    (time) int64 392B 1 16 31 46 61 76 ... 646 661 676 691 706 721